# NumPy Dojo — Block 6: ML Algorithms

Covers: 2D convolution, K-means, SVD/low-rank approximation, Precision/Recall/F1, L1/L2 regularization.

Dataset: **sklearn digits** (images for conv) + synthetic clusters (K-means) + embeddings (SVD).

These are complete algorithms, not primitives — expect 15–25 min per problem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(42)

digits = load_digits()
images = digits.images.astype(float)  # (1797, 8, 8)
X_flat = digits.data.astype(float)    # (1797, 64)
y = digits.target

# Synthetic clusters for K-means
centers_true = np.array([[-4,-4],[4,-4],[0,4],[-4,4],[4,4]], dtype=float)
X_clust = np.vstack([
    np.random.randn(80, 2) + c for c in centers_true
])  # (400, 2)
y_clust = np.repeat(np.arange(5), 80)

print(f'images: {images.shape}')
print(f'X_clust: {X_clust.shape}')

---
## P1 — 2D Convolution from Scratch

Implement 2D convolution using only NumPy — no `scipy.signal`, no `cv2`.

1. Implement **single-image, single-kernel** conv2d: `(H, W)` × `(kH, kW)` → `(H', W')` (valid padding)
2. Extend to **batched multi-channel**: `(N, C_in, H, W)` × `(C_out, C_in, kH, kW)` → `(N, C_out, H', W')`
3. Apply a Sobel edge-detection kernel to digit images and visualize

Hint for the batched version: nested loops over spatial positions is fine — focus on correctness first, then think about whether any axes can be vectorized.

In [ ]:
def conv2d_single(img, kernel):
    """
    img:    (H, W)
    kernel: (kH, kW)
    Returns (H - kH + 1, W - kW + 1) — valid convolution
    """
    H, W = img.shape
    kH, kW = kernel.shape
    out_H, out_W = H - kH + 1, W - kW + 1
    out = np.zeros((out_H, out_W))
    
    # TODO: slide kernel over img
    # out[i, j] = sum of elementwise product of kernel and img[i:i+kH, j:j+kW]
    # Use nested loops over i and j (spatial positions)
    pass


def conv2d_batched(X, W_conv):
    """
    X:      (N, C_in, H, W)
    W_conv: (C_out, C_in, kH, kW)
    Returns (N, C_out, H', W')
    """
    N, C_in, H, W = X.shape
    C_out, _, kH, kW = W_conv.shape
    out_H, out_W = H - kH + 1, W - kW + 1
    out = np.zeros((N, C_out, out_H, out_W))
    
    # TODO: loop over spatial positions
    # For each (i,j): out[:, :, i, j] = sum over c_in of X[:, c_in, i:i+kH, j:j+kW] * W_conv[:, c_in, :, :]
    # Think about what shapes you're multiplying and which axes to sum over
    pass


# Sobel kernels for edge detection
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = np.array([[-1,-2,-1], [ 0, 0, 0], [ 1, 2, 1]], dtype=float)

edges_x = conv2d_single(images[0], sobel_x)
edges_y = conv2d_single(images[0], sobel_y)
edge_mag = np.sqrt(edges_x**2 + edges_y**2)

In [ ]:
# --- ASSERTS ---

# Single conv
img_test = np.arange(25, dtype=float).reshape(5, 5)
k = np.array([[1,0],[0,-1]], dtype=float)
out = conv2d_single(img_test, k)
assert out.shape == (4, 4), f'Expected (4,4), got {out.shape}'
# out[0,0] = 0*1 + 1*0 + 5*0 + 6*(-1) = -6
assert np.isclose(out[0, 0], 0 - 6), f'Expected -6, got {out[0,0]}'

# Batched conv
N_t, C_in_t, H_t, W_t = 4, 2, 6, 6
C_out_t, kH_t, kW_t = 3, 3, 3
X_t = np.random.randn(N_t, C_in_t, H_t, W_t)
W_t_conv = np.random.randn(C_out_t, C_in_t, kH_t, kW_t)
out_batched = conv2d_batched(X_t, W_t_conv)
assert out_batched.shape == (N_t, C_out_t, H_t - kH_t + 1, W_t - kW_t + 1), f'Batched shape wrong: {out_batched.shape}'

# Verify batched against single
for c_out in range(C_out_t):
    for c_in in range(C_in_t):
        expected = conv2d_single(X_t[0, c_in], W_t_conv[c_out, c_in])
    # sum over c_in channels
    expected_sum = sum(conv2d_single(X_t[0, c_in], W_t_conv[c_out, c_in]) for c_in in range(C_in_t))
    assert np.allclose(out_batched[0, c_out], expected_sum, atol=1e-8), f'Batched mismatch at c_out={c_out}'

print('P1 PASSED ✓')

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
axes[0].imshow(images[0], cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(edges_x, cmap='RdBu'); axes[1].set_title('Sobel-X')
axes[2].imshow(edges_y, cmap='RdBu'); axes[2].set_title('Sobel-Y')
axes[3].imshow(edge_mag, cmap='hot'); axes[3].set_title('Edge magnitude')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## P2 — K-Means from Scratch

Implement K-Means clustering:

1. Initialize: pick k random data points as starting centroids (use index sampling, not `np.random.choice` on the data directly)
2. Assign each point to nearest centroid — use your pairwise L2 from notebook 03
3. Update centroids as mean of assigned points
4. Handle empty clusters: reassign to a random data point
5. Stop when assignments don't change or after max_iter

Return final centroids, assignments, and inertia (sum of squared distances to assigned centroid).

In [ ]:
def pairwise_l2(A, B):
    """(N,D) x (M,D) -> (N,M) — copy from notebook 03 or re-implement"""
    # TODO
    pass

def kmeans(X, k, max_iter=100, seed=0):
    """
    X: (N, D)
    Returns:
        centroids:   (k, D)
        assignments: (N,) integer in [0, k)
        inertia:     scalar — sum of squared distances to assigned centroid
    """
    np.random.seed(seed)
    N, D = X.shape
    
    # Step 1: TODO — initialize k centroids by random sampling without replacement
    
    assignments = np.zeros(N, dtype=int)
    
    for iteration in range(max_iter):
        # Step 2: TODO — assign each point to nearest centroid
        # Use pairwise_l2(X, centroids), then argmin over centroid axis
        
        # Check convergence
        # TODO: if assignments didn't change, break
        
        # Step 3: TODO — update centroids
        # For each cluster, mean of assigned points
        # Don't forget: handle empty clusters (no points assigned)
        pass
    
    # Compute inertia
    # TODO: for each point, squared distance to its assigned centroid, sum
    inertia = None
    
    return centroids, assignments, inertia

centroids, assignments, inertia = kmeans(X_clust, k=5)

In [ ]:
# --- ASSERTS ---
assert centroids.shape == (5, 2)
assert assignments.shape == (400,)
assert set(assignments) == {0, 1, 2, 3, 4}, 'All 5 clusters should be non-empty'

# Quality check: assigned centroids should be close to true centers
# Match each found centroid to nearest true center
from itertools import permutations
best_err = np.inf
for perm in permutations(range(5)):
    err = np.mean([np.linalg.norm(centroids[perm[i]] - centers_true[i]) for i in range(5)])
    best_err = min(best_err, err)
assert best_err < 1.5, f'Centroids far from true centers (mean dist={best_err:.2f})'

assert inertia is not None and inertia > 0

print(f'P2 PASSED ✓  inertia={inertia:.2f}, best centroid error={best_err:.3f}')

plt.figure(figsize=(6, 5))
colors = ['steelblue','tomato','seagreen','darkorange','purple']
for i in range(5):
    mask = assignments == i
    plt.scatter(X_clust[mask, 0], X_clust[mask, 1], alpha=0.4, s=15, c=colors[i])
    plt.scatter(*centroids[i], marker='X', s=200, c=colors[i], edgecolors='black', linewidths=1.5)
plt.scatter(centers_true[:, 0], centers_true[:, 1], marker='*', s=300, c='gold', edgecolors='black', linewidths=1, label='True centers')
plt.legend(); plt.title('K-Means result'); plt.tight_layout(); plt.show()

---
## P3 — SVD and Low-Rank Approximation

SVD decomposes `A = U Σ Vᵀ` where U and V are orthogonal, Σ is diagonal with singular values.

1. Decompose a digit image using `np.linalg.svd`
2. Reconstruct using only top-k singular values/vectors — visualize quality vs k
3. Compute **explained variance** per singular value
4. Implement **truncated matrix factorization**: given a rating matrix with missing values, fill using SVD on the observed entries
5. Verify: SVD-based PCA vs eigendecomposition-based PCA from notebook 03 should give same projections

In [ ]:
img = images[0]  # (8, 8)

def svd_reconstruct(img, k):
    """
    img: (H, W)
    Reconstruct using top-k singular values
    Returns (H, W)
    """
    # TODO: np.linalg.svd with full_matrices=False
    # U: (H, min(H,W)), s: (min(H,W),), Vt: (min(H,W), W)
    # Reconstruct: U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    pass

def svd_explained_variance(img):
    """
    Returns explained variance ratio for each singular value: s_i^2 / sum(s^2)
    """
    # TODO: np.linalg.svd, then compute ratios
    pass

def pca_via_svd(X, k):
    """
    X: (N, D) — center first, then use SVD on centered X
    Returns X_proj (N, k)
    Relationship: SVD of X_centered = U S Vt, then X_proj = X_centered @ Vt[:k].T
    """
    # TODO: center X, compute SVD, project
    # Note: Vt rows are the principal components (equivalent to eigenvectors of covariance matrix)
    pass

evr = svd_explained_variance(img)

In [ ]:
# --- ASSERTS ---

# Full-rank SVD should reconstruct exactly
k_full = min(img.shape)
recon_full = svd_reconstruct(img, k=k_full)
assert np.allclose(recon_full, img, atol=1e-6), 'Full-rank SVD should reconstruct exactly'

# Rank-1 should be much worse than rank-4
err_k1 = np.linalg.norm(svd_reconstruct(img, 1) - img, 'fro')
err_k4 = np.linalg.norm(svd_reconstruct(img, 4) - img, 'fro')
assert err_k4 < err_k1, 'Higher rank should give lower reconstruction error'

# Explained variance should sum to 1
assert np.isclose(evr.sum(), 1.0, atol=1e-6)
assert np.all(evr >= 0)
assert evr[0] >= evr[1], 'Should be sorted descending'

# SVD-based PCA should give same projections as eig-based PCA (up to sign)
# (signs of eigenvectors are arbitrary — compare absolute values)
X_hd = X_flat[:100]  # small subset
X_proj_svd = pca_via_svd(X_hd, k=5)
assert X_proj_svd.shape == (100, 5)

print(f'P3 PASSED ✓')
print(f'  Rank-1 reconstruction error: {err_k1:.3f}')
print(f'  Rank-4 reconstruction error: {err_k4:.3f}')
print(f'  Top-3 SVs explain: {evr[:3].sum()*100:.1f}% of variance')

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
for ki, k in enumerate([1, 2, 4, 8]):
    axes[ki+1].imshow(svd_reconstruct(img, k), cmap='gray')
    axes[ki+1].set_title(f'k={k}')
for ax in axes: ax.axis('off')
plt.suptitle('SVD reconstruction quality'); plt.tight_layout(); plt.show()

---
## P4 — Precision, Recall, F1

Implement multi-class metrics from scratch — no sklearn.

Given predictions and true labels:
1. **Per-class Precision**: TP / (TP + FP)
2. **Per-class Recall**: TP / (TP + FN)
3. **Per-class F1**: harmonic mean of P and R
4. **Macro average**: mean of per-class values
5. **Weighted average**: weighted by class support (count)

Handle the edge case: if a class has no predictions or no true positives, P or R = 0.

In [ ]:
# Simulate predictions from a decent but imperfect model
np.random.seed(1)
y_true = y[:300]
# Mostly correct, some noise
y_pred = y_true.copy()
noise_idx = np.random.choice(300, 60, replace=False)
y_pred[noise_idx] = np.random.randint(0, 10, 60)

def precision_recall_f1(y_true, y_pred, num_classes):
    """
    Returns:
        precision: (num_classes,)
        recall:    (num_classes,)
        f1:        (num_classes,)
        support:   (num_classes,) — true count per class
    """
    precision = np.zeros(num_classes)
    recall    = np.zeros(num_classes)
    f1        = np.zeros(num_classes)
    support   = np.zeros(num_classes, dtype=int)
    
    for c in range(num_classes):
        # TODO: TP, FP, FN using boolean masks
        # TP: predicted c AND true c
        # FP: predicted c AND true != c
        # FN: true c AND predicted != c
        pass
    
    return precision, recall, f1, support

def macro_avg(precision, recall, f1):
    """Unweighted mean over classes"""
    # TODO
    pass

def weighted_avg(precision, recall, f1, support):
    """Weighted by support"""
    # TODO
    pass

prec, rec, f1_scores, support = precision_recall_f1(y_true, y_pred, num_classes=10)

In [ ]:
# --- ASSERTS (compare vs sklearn) ---
from sklearn.metrics import precision_recall_fscore_support
sk_prec, sk_rec, sk_f1, sk_sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(10)), zero_division=0)

assert np.allclose(prec, sk_prec, atol=1e-6), f'Precision mismatch: {prec} vs {sk_prec}'
assert np.allclose(rec,  sk_rec,  atol=1e-6), f'Recall mismatch'
assert np.allclose(f1_scores, sk_f1, atol=1e-6), f'F1 mismatch'
assert np.all(support == sk_sup)

mac_p, mac_r, mac_f = macro_avg(prec, rec, f1_scores)
wt_p, wt_r, wt_f   = weighted_avg(prec, rec, f1_scores, support)

from sklearn.metrics import precision_recall_fscore_support as prfs
sk_mac = prfs(y_true, y_pred, average='macro', zero_division=0)
sk_wt  = prfs(y_true, y_pred, average='weighted', zero_division=0)

assert np.isclose(mac_f, sk_mac[2], atol=1e-5), f'Macro F1: {mac_f:.4f} vs {sk_mac[2]:.4f}'
assert np.isclose(wt_f,  sk_wt[2],  atol=1e-5), f'Weighted F1: {wt_f:.4f} vs {sk_wt[2]:.4f}'

print('P4 PASSED ✓')
print(f'  Macro F1:    {mac_f:.4f}')
print(f'  Weighted F1: {wt_f:.4f}')

---
## P5 — L1/L2 Regularization

Regularization modifies the loss: `L_total = L_task + λ * R(W)`

Implement both regularization terms and their gradients, then wire into the MLP training loop from notebook 04.

- **L2 (Ridge)**: `R(W) = 0.5 * ||W||²_F`, gradient = `W`
- **L1 (Lasso)**: `R(W) = ||W||_1 = sum(|W|)`, gradient = `sign(W)`

Note: regularization is applied only to weight matrices, **not** biases.

In [ ]:
def l2_reg(params, lam):
    """
    params: dict with W1, b1, W2, b2
    Returns: (loss_term, grad_dict) — grad has same keys as params
    Only regularize W1, W2 — not biases
    """
    # TODO: loss = 0.5 * lam * (||W1||^2 + ||W2||^2)
    # TODO: grads = {W1: lam*W1, b1: zeros, W2: lam*W2, b2: zeros}
    pass

def l1_reg(params, lam):
    """
    Returns: (loss_term, grad_dict)
    Gradient of |W| = sign(W) (subgradient at 0 = 0)
    """
    # TODO
    pass

# Verify: gradient of L2 should equal lam * W
# Verify: gradient of L1 should be lam * sign(W)

def total_loss_and_grads(X, y, params, lam=1e-3, reg='l2'):
    """
    Combines task loss + regularization.
    Returns: total_loss, combined grads
    """
    # TODO: compute task loss + grads (copy forward/backward from notebook 04)
    # TODO: compute regularization loss + grads
    # TODO: add them element-wise
    pass

In [ ]:
# --- ASSERTS ---
D_t, H_t, C_t = 64, 32, 10
np.random.seed(0)
params_t = {
    'W1': np.random.randn(H_t, D_t) * 0.1,
    'b1': np.zeros(H_t),
    'W2': np.random.randn(C_t, H_t) * 0.1,
    'b2': np.zeros(C_t)
}
lam = 1e-2

l2_loss, l2_grads = l2_reg(params_t, lam)
l1_loss, l1_grads = l1_reg(params_t, lam)

# L2 loss
expected_l2 = 0.5 * lam * (np.sum(params_t['W1']**2) + np.sum(params_t['W2']**2))
assert np.isclose(l2_loss, expected_l2, rtol=1e-5), f'L2 loss: {l2_loss} vs {expected_l2}'

# L2 gradient for W1
assert np.allclose(l2_grads['W1'], lam * params_t['W1']), 'L2 grad W1 wrong'
assert np.allclose(l2_grads['b1'], 0), 'L2 bias grad should be 0'

# L1 loss
expected_l1 = lam * (np.sum(np.abs(params_t['W1'])) + np.sum(np.abs(params_t['W2'])))
assert np.isclose(l1_loss, expected_l1, rtol=1e-5), f'L1 loss: {l1_loss} vs {expected_l1}'

# L1 gradient: sign(W)
assert np.allclose(l1_grads['W1'], lam * np.sign(params_t['W1'])), 'L1 grad W1 wrong'
assert np.allclose(l1_grads['b2'], 0), 'L1 bias grad should be 0'

print('P5 PASSED ✓')

# Visualize: effect of L2 on weight magnitude over training
# Compare unregularized vs L2 regularized training on digits
print('\nDemonstrating regularization effect on weight norms...')

# You should see W norms shrink more with stronger lambda
for lam_val in [0, 1e-4, 1e-2]:
    W = np.random.randn(32, 64) * 0.1
    for _ in range(50):
        grad = np.random.randn(*W.shape) * 0.01  # simulate task grad
        reg_grad = lam_val * W
        W -= 0.01 * (grad + reg_grad)
    print(f'  λ={lam_val:.0e}  → final ||W||_F = {np.linalg.norm(W, "fro"):.4f}')